# Step 6 + SBERT-FT + BGE-M3 reranker FT + LLM XGB fusion

Notebook này chỉ fuse public rankings/scored candidates đã có. Không train, không inference, không cần GPU.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import zipfile
from typing import Any

import pandas as pd

DATA_ROOT = Path('/kaggle/input/datasets/bowboochua9/stnhdscduaiti26')
WEIGHT_ROOT = Path('/kaggle/input/datasets/mphatfromuit/dsc2026-baseline/DSC26_weight_report')
OUTPUT_DIR = Path('/kaggle/working/step6_sbertft_bgem3rerankft_llmxgb')

STEP6_RANKINGS = DATA_ROOT / 'step7/step6/rankings/aiteamvn_vietnamese_reranker_finetuned/public_rankings_step6_fused.jsonl'
SBERT_RANKINGS = WEIGHT_ROOT / 'legalir_sbert_cl/public_submission/public_ranked_contexts.csv'
CE_RANKINGS = WEIGHT_ROOT / 'legalir_ce_listwise/public_submission/public_official_bm25_sbert_ce_rankings.jsonl'
XGB_SCORED_CANDIDATES = WEIGHT_ROOT / 'legalir_LLM_XGB_reranker/public_submission/public_official_xgb_scored_candidates.parquet'
XGB_SUBMISSION = WEIGHT_ROOT / 'legalir_LLM_XGB_reranker/public_submission/submission.json'

required = {
    'step6_rankings': STEP6_RANKINGS,
    'sbert_rankings': SBERT_RANKINGS,
    'ce_bge_m3_reranker_v2_rankings': CE_RANKINGS,
    'xgb_scored_candidates': XGB_SCORED_CANDIDATES,
    'xgb_submission': XGB_SUBMISSION,
}
missing = {name: str(path) for name, path in required.items() if not path.exists()}
print('DATA_ROOT:', DATA_ROOT)
print('WEIGHT_ROOT:', WEIGHT_ROOT)
print(json.dumps({k: str(v) for k, v in required.items()}, ensure_ascii=False, indent=2))
if missing:
    print('Missing required paths:')
    print(json.dumps(missing, ensure_ascii=False, indent=2))
    raise FileNotFoundError('Missing fixed input artifacts. No fallback is used.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
def read_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def iter_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def unique_keep_order(items):
    seen = set()
    out = []
    for item in items:
        if item is None:
            continue
        if isinstance(item, dict):
            item = item.get('doc_id') or item.get('document_id') or item.get('context_id') or item.get('id')
        item = str(item)
        if item and item not in seen:
            seen.add(item)
            out.append(item)
    return out


QID_KEYS = ['query_id', 'qid', 'question_id', 'id']
DOC_LIST_KEYS = ['fused_doc_ids', 'doc_ids', 'document_ids', 'ranked_doc_ids', 'reranked_doc_ids', 'top_doc_ids', 'top_docs', 'answer', 'answers', 'predictions', 'docs', 'candidates']


def extract_qid(row: dict[str, Any]) -> str:
    for key in QID_KEYS:
        if key in row and row[key] is not None:
            return str(row[key])
    raise KeyError({'message': 'Cannot infer query id field', 'keys': sorted(row.keys())})


def normalize_doc_value(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            return normalize_doc_value(json.loads(text))
        except json.JSONDecodeError:
            return unique_keep_order(text.replace(';', ',').split(','))
    if isinstance(value, dict):
        if 'answer' in value:
            return normalize_doc_value(value['answer'])
        return unique_keep_order([value])
    if isinstance(value, list):
        return unique_keep_order(value)
    return unique_keep_order([value])


def extract_docs(row: dict[str, Any]) -> list[str]:
    for key in DOC_LIST_KEYS:
        if key in row:
            docs = normalize_doc_value(row[key])
            if docs:
                return docs
    raise KeyError({'message': 'Cannot infer ranked docs field', 'keys': sorted(row.keys()), 'sample': row})


def load_jsonl_rankings(path: Path, label: str) -> dict[str, list[str]]:
    rankings = {}
    first = None
    for row in iter_jsonl(path):
        if first is None:
            first = row
        rankings[extract_qid(row)] = extract_docs(row)
    first_qid = next(iter(rankings)) if rankings else None
    print(f'{label}: rows={len(rankings)}')
    print(f'{label}: sample keys={sorted(first.keys()) if first else []}')
    print(f'{label}: first_qid={first_qid}, first_docs={rankings[first_qid][:8] if first_qid else []}')
    return rankings


def load_sbert_csv(path: Path) -> dict[str, list[str]]:
    rankings = {}
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        print('sbert columns:', reader.fieldnames)
        for row in reader:
            qid = str(row.get('qid') or row.get('query_id') or row.get('question_id'))
            if qid == 'None':
                raise KeyError({'message': 'Cannot infer SBERT qid column', 'columns': reader.fieldnames})
            for col in ['top20_rerank', 'top20', 'answer', 'answers', 'doc_ids']:
                if row.get(col):
                    rankings[qid] = normalize_doc_value(row[col])
                    break
            else:
                raise KeyError({'message': 'Cannot infer SBERT docs column', 'columns': reader.fieldnames})
    first_qid = next(iter(rankings))
    print('sbert: rows=', len(rankings), 'first_qid=', first_qid, 'first_docs=', rankings[first_qid][:8])
    return rankings


def load_submission_rankings(path: Path, label: str) -> dict[str, list[str]]:
    obj = read_json(path)
    rankings = {}
    for qid, item in obj.items():
        if not isinstance(item, dict) or 'answer' not in item:
            raise ValueError({'label': label, 'qid': qid, 'item': item})
        rankings[str(qid)] = normalize_doc_value(item['answer'])
    first_qid = next(iter(rankings))
    print(f'{label}: rows={len(rankings)}, first_qid={first_qid}, first_docs={rankings[first_qid][:8]}')
    return rankings


def find_column(df: pd.DataFrame, exact_candidates: list[str], contains_candidates: list[str], label: str) -> str:
    lower_to_col = {str(c).lower(): c for c in df.columns}
    for cand in exact_candidates:
        if cand.lower() in lower_to_col:
            return lower_to_col[cand.lower()]
    for col in df.columns:
        low = str(col).lower()
        if any(token in low for token in contains_candidates):
            return col
    raise KeyError({'message': f'Cannot infer {label} column', 'columns': list(map(str, df.columns))})


def load_xgb_scored_candidates(path: Path) -> tuple[dict[str, list[str]], dict[str, Any]]:
    df = pd.read_parquet(path)
    print('xgb scored candidates shape:', df.shape)
    print('xgb columns:', list(map(str, df.columns)))
    print(df.head(3).to_string())
    qid_col = find_column(df, ['query_id', 'qid', 'question_id'], ['query'], 'qid')
    doc_col = find_column(df, ['candidate_id', 'doc_id', 'document_id', 'context_id', 'candidate_doc_id'], ['doc', 'candidate'], 'doc_id')
    score_col = find_column(df, ['xgb_llm_score', 'xgb_score', 'score', 'prediction', 'pred', 'rank_score', 'final_score'], ['xgb_llm_score', 'score', 'pred'], 'score')
    work = df[[qid_col, doc_col, score_col]].copy()
    work[qid_col] = work[qid_col].astype(str)
    work[doc_col] = work[doc_col].astype(str)
    work[score_col] = pd.to_numeric(work[score_col], errors='coerce')
    if work[score_col].isna().any():
        raise ValueError({'message': 'XGB score column contains NaN after numeric conversion', 'score_col': str(score_col), 'num_nan': int(work[score_col].isna().sum())})
    work = work.sort_values([qid_col, score_col, doc_col], ascending=[True, False, True])
    rankings = {}
    for qid, group in work.groupby(qid_col, sort=False):
        rankings[str(qid)] = unique_keep_order(group[doc_col].tolist())
    meta = {'qid_col': str(qid_col), 'doc_col': str(doc_col), 'score_col': str(score_col), 'num_rows': int(len(work)), 'num_queries': int(len(rankings))}
    first_qid = next(iter(rankings))
    print('xgb inferred meta:', meta)
    print('xgb first_qid=', first_qid, 'first_docs=', rankings[first_qid][:8])
    return rankings, meta


def rrf_fuse(named_rankings: list[tuple[str, list[str], float]], rrf_k: int = 60, depth: int = 100) -> list[str]:
    scores = {}
    tie = {}
    source_order = 0
    for source_name, docs, weight in named_rankings:
        if weight <= 0:
            continue
        for rank, doc_id in enumerate(docs[:depth], start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + weight / (rrf_k + rank)
            tie.setdefault(doc_id, (rank, source_order, doc_id))
        source_order += 1
    return sorted(scores, key=lambda doc_id: (-scores[doc_id], tie[doc_id]))


def validate_submission(submission: dict[str, Any], qids: list[str]) -> dict[str, Any]:
    issues = []
    qid_set = set(qids)
    sub_set = set(submission.keys())
    if qid_set != sub_set:
        issues.append({'type': 'query_id_mismatch', 'missing': sorted(qid_set - sub_set)[:10], 'extra': sorted(sub_set - qid_set)[:10]})
    length_dist = {}
    for qid in qids:
        answer = submission.get(qid, {}).get('answer') if isinstance(submission.get(qid), dict) else None
        if not isinstance(answer, list):
            issues.append({'type': 'answer_not_list', 'qid': qid})
            continue
        length_dist[str(len(answer))] = length_dist.get(str(len(answer)), 0) + 1
        if not (1 <= len(answer) <= 5):
            issues.append({'type': 'bad_answer_length', 'qid': qid, 'length': len(answer)})
        if len(set(answer)) != len(answer):
            issues.append({'type': 'duplicate_doc_id', 'qid': qid})
        if any(not isinstance(x, str) for x in answer):
            issues.append({'type': 'non_string_doc_id', 'qid': qid})
    return {'num_public_queries': len(qids), 'num_submission_queries': len(submission), 'answer_length_distribution': dict(sorted(length_dist.items())), 'num_errors': len(issues), 'issues': issues[:50]}


def zip_submission(submission_json: Path, submission_zip: Path) -> None:
    with zipfile.ZipFile(submission_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(submission_json, arcname='submission.json')
    with zipfile.ZipFile(submission_zip, 'r') as zf:
        names = zf.namelist()
    if names != ['submission.json']:
        raise ValueError(f'Bad zip entries: {names}')


In [ ]:
step6 = load_jsonl_rankings(STEP6_RANKINGS, 'step6')
sbert = load_sbert_csv(SBERT_RANKINGS)
ce = load_jsonl_rankings(CE_RANKINGS, 'ce_bge_m3_reranker_v2')
xgb, xgb_meta = load_xgb_scored_candidates(XGB_SCORED_CANDIDATES)
xgb_submission_rankings = load_submission_rankings(XGB_SUBMISSION, 'xgb_llm_submission')

qids = list(step6.keys())
missing = {
    'sbert_missing': [qid for qid in qids if qid not in sbert][:20],
    'ce_missing': [qid for qid in qids if qid not in ce][:20],
    'xgb_missing': [qid for qid in qids if qid not in xgb][:20],
    'xgb_submission_missing': [qid for qid in qids if qid not in xgb_submission_rankings][:20],
}
if any(missing.values()):
    print(json.dumps(missing, ensure_ascii=False, indent=2))
    raise ValueError('Some query IDs are missing from one or more branches.')

xgb_parquet_vs_submission_changed = 0
for qid in qids:
    if xgb[qid][:5] != xgb_submission_rankings[qid][:5]:
        xgb_parquet_vs_submission_changed += 1
print('qids:', len(qids))
print('xgb_parquet_top5_differs_from_submission_queries:', xgb_parquet_vs_submission_changed)

In [ ]:
def make_rrf_submission(step6_weight: float, sbert_weight: float, ce_weight: float, xgb_weight: float, *, depth: int = 100, rrf_k: int = 60) -> dict[str, dict[str, list[str]]]:
    out = {}
    for qid in qids:
        ranked = rrf_fuse([
            ('step6', step6[qid], step6_weight),
            ('sbertft', sbert[qid], sbert_weight),
            ('ce_bge_m3_reranker_v2_ft', ce[qid], ce_weight),
            ('llm_xgb', xgb[qid], xgb_weight),
        ], rrf_k=rrf_k, depth=depth)
        out[qid] = {'answer': ranked[:5]}
    return out


def submission_from_rankings(rankings: dict[str, list[str]]) -> dict[str, dict[str, list[str]]]:
    return {qid: {'answer': rankings[qid][:5]} for qid in qids}


def audit_vs_control(candidate: dict[str, Any], control: dict[str, Any], qids: list[str]) -> dict[str, Any]:
    changed = 0
    overlap_control = 0
    overlap_ce = 0
    overlap_xgb = 0
    for qid in qids:
        docs = candidate[qid]['answer']
        cdocs = control[qid]['answer']
        changed += docs != cdocs
        overlap_control += len(set(docs) & set(cdocs))
        overlap_ce += len(set(docs) & set(ce[qid][:5]))
        overlap_xgb += len(set(docs) & set(xgb_submission_rankings[qid][:5]))
    n = len(qids)
    return {'changed_queries_vs_baselinecur': changed, 'avg_overlap_baselinecur_top5': overlap_control / n, 'avg_overlap_ce_top5': overlap_ce / n, 'avg_overlap_xgb_submission_top5': overlap_xgb / n}


candidate_configs = [
    {'name': 'baseline_rrf_sbert0p60_control', 'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.6, 'ce_weight': 0.0, 'xgb_weight': 0.0},
    {'name': 'ce_only_control', 'kind': 'rrf', 'step6_weight': 0.0, 'sbert_weight': 0.0, 'ce_weight': 1.0, 'xgb_weight': 0.0},
    {'name': 'xgb_llm_only_control', 'kind': 'xgb_submission'},
    {'name': 'baseline_plus_xgb_w0p15', 'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.6, 'ce_weight': 0.0, 'xgb_weight': 0.15},
    {'name': 'baseline_plus_xgb_w0p25', 'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.6, 'ce_weight': 0.0, 'xgb_weight': 0.25},
    {'name': 'baseline_plus_xgb_w0p40', 'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.6, 'ce_weight': 0.0, 'xgb_weight': 0.40},
    {'name': 'baseline_plus_xgb_w0p60', 'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.6, 'ce_weight': 0.0, 'xgb_weight': 0.60},
    {'name': 'baseline_plus_ce_w0p25_plus_xgb_w0p25', 'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.6, 'ce_weight': 0.25, 'xgb_weight': 0.25},
    {'name': 'full_ce0p25_xgb0p40', 'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.6, 'ce_weight': 0.25, 'xgb_weight': 0.40},
    {'name': 'full_ce0p40_xgb0p60', 'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.6, 'ce_weight': 0.40, 'xgb_weight': 0.60},
]

control = make_rrf_submission(1.0, 0.6, 0.0, 0.0)
reports = []
for cfg in candidate_configs:
    name = cfg['name']
    cand_dir = OUTPUT_DIR / 'candidates' / name
    cand_dir.mkdir(parents=True, exist_ok=True)
    if cfg['kind'] == 'xgb_submission':
        sub = submission_from_rankings(xgb_submission_rankings)
    else:
        sub = make_rrf_submission(cfg['step6_weight'], cfg['sbert_weight'], cfg['ce_weight'], cfg['xgb_weight'])
    validation = validate_submission(sub, qids)
    if validation['num_errors']:
        raise ValueError({'candidate': name, 'validation': validation})
    write_json(cand_dir / 'submission.json', sub)
    write_json(cand_dir / 'submission_validation.json', validation)
    zip_submission(cand_dir / 'submission.json', cand_dir / 'submission.zip')
    info = dict(cfg)
    info.update(audit_vs_control(sub, control, qids))
    info['submission_zip'] = str(cand_dir / 'submission.zip')
    write_json(cand_dir / 'candidate_info.json', info)
    reports.append(info)

default_candidate = 'baseline_plus_xgb_w0p25'
default_json = OUTPUT_DIR / 'candidates' / default_candidate / 'submission.json'
write_json(OUTPUT_DIR / 'submission.json', read_json(default_json))
zip_submission(OUTPUT_DIR / 'submission.json', OUTPUT_DIR / 'submission.zip')

run_report = {
    'status': 'ok',
    'method': 'weighted_rrf_step6_sbertft_bge_m3_reranker_v2_ft_llm_xgb',
    'default_candidate': default_candidate,
    'baseline_public_recall_to_beat': 0.9288333333333333,
    'reported_xgb_llm_public_recall_reference': 0.9227,
    'xgb_meta': xgb_meta,
    'xgb_parquet_top5_differs_from_submission_queries': xgb_parquet_vs_submission_changed,
    'input_files': {name: str(path) for name, path in required.items()},
    'input_sha256': {name: sha256_file(path) for name, path in required.items()},
    'candidate_audit': reports,
}
write_json(OUTPUT_DIR / 'run_report.json', run_report)
print(json.dumps(run_report, ensure_ascii=False, indent=2))